In [1]:
import os, sys

PROJECT_ROOT = '/scratch/jq2uw/derm_vlms'
GPT53_DIR = os.path.join(PROJECT_ROOT, 'gpt53')

if GPT53_DIR not in sys.path:
    sys.path.insert(0, GPT53_DIR)

sys.path.insert(0, PROJECT_ROOT)
from tokens import AZURE_GPT53_API_KEY

from utils import init_client, predict_image, parse_label

print('Initialising Azure GPT-5.3 client...')
client = init_client(api_key=AZURE_GPT53_API_KEY)
print('Client ready.')

Initialising Azure GPT-5.3 client...
Client ready.


In [2]:
import pandas as pd
from pathlib import Path

sys.path.insert(0, PROJECT_ROOT)
from data_utils import prepare_all_lesions

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
IMAGES_DIR = os.path.join(RESULTS_DIR, 'images')
OUT_PATH = os.path.join(RESULTS_DIR, 'gpt53_predictions_all.csv')

df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data_share', 'midas_share.parquet'))
print(f'Loaded {len(df)} rows')
print(f'y3 distribution:\n{df["y3"].value_counts()}')
print(f'y16 distribution:\n{df["y16"].value_counts()}')

df_all = prepare_all_lesions(df, data_dir=DATA_DIR, output_dir=IMAGES_DIR)
print(f'\nTotal prediction rows: {len(df_all)}')
df_all.head(10)

Loaded 3357 rows
y3 distribution:
y3
malignant    1391
benign       1322
other         644
Name: count, dtype: int64
Sampled 10 lesions (5 per class) -> 30 rows
Classes: {'benign': 15, 'malignant': 15}
Images saved to: /scratch/jq2uw/derm_vlms/results/images


,id,ground_truth,image_mode,image_path,original_image_name,lesion_id
0,1_photo,benign,photo,/scratch/jq2uw/derm_vlms/results/images/1_phot...,s-prd-667118134.jpg,534_left lower leg_no
1,1_dscope,benign,dscope,/scratch/jq2uw/derm_vlms/results/images/1_dsco...,s-prd-667118139.jpg,534_left lower leg_no
2,1_combined,benign,combined,/scratch/jq2uw/derm_vlms/results/images/1_comb...,s-prd-667118134.jpg; s-prd-667118139.jpg,534_left lower leg_no
3,2_photo,benign,photo,/scratch/jq2uw/derm_vlms/results/images/2_phot...,s-prd-505211076.jpg,215_central chest _no
4,2_dscope,benign,dscope,/scratch/jq2uw/derm_vlms/results/images/2_dsco...,s-prd-505211307.jpg,215_central chest _no


In [3]:
from PIL import Image
from tqdm import tqdm

PROMPT = ("You are an expert dermatologist. Please describe using dermatological "
          "terms to describe the appearance of this lesion. "
          "Please give the top 3 diagnoses in your differential.")

CHECKPOINT_EVERY = 10
COL_ORDER = ['id', 'ground_truth', 'y16', 'y16_description', 'image_mode',
             'describe_then_classify', 'image_path', 'original_image_name', 'lesion_id']

# --- Resume logic ---
if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH)
    done_ids = set(existing_df['id'].tolist())
    print(f'Resuming: {len(done_ids)} predictions already completed')
else:
    done_ids = set()

pending = df_all[~df_all['id'].isin(done_ids)]
print(f'{len(pending)} predictions remaining out of {len(df_all)} total')

# --- Prediction loop with checkpointing ---
batch = []
lesions_in_batch = set()

for _, row in tqdm(pending.iterrows(), total=len(pending)):
    try:
        image = Image.open(row['image_path']).convert('RGB')
    except Exception as e:
        print(f'[SKIP] {row["id"]}: {e}')
        continue

    try:
        response = predict_image(client, image, prompt=PROMPT)
    except Exception as e:
        print(f'[ERR] {row["id"]}: {e}')
        response = f'ERROR: {e}'

    batch.append({
        'id': row['id'],
        'ground_truth': row['ground_truth'],
        'y16': row['y16'],
        'y16_description': row['y16_description'],
        'image_mode': row['image_mode'],
        'describe_then_classify': response,
        'image_path': row['image_path'],
        'original_image_name': row['original_image_name'],
        'lesion_id': row['lesion_id'],
    })

    lesions_in_batch.add(row['lesion_id'])

    if len(lesions_in_batch) >= CHECKPOINT_EVERY:
        batch_df = pd.DataFrame(batch)[COL_ORDER]
        header = not os.path.exists(OUT_PATH)
        batch_df.to_csv(OUT_PATH, mode='a', header=header, index=False)
        print(f'\n[Checkpoint] +{len(batch)} rows ({len(lesions_in_batch)} lesions)')
        batch = []
        lesions_in_batch = set()

if batch:
    batch_df = pd.DataFrame(batch)[COL_ORDER]
    header = not os.path.exists(OUT_PATH)
    batch_df.to_csv(OUT_PATH, mode='a', header=header, index=False)
    print(f'\n[Final] +{len(batch)} rows ({len(lesions_in_batch)} lesions)')

print(f'\nAll predictions saved to {OUT_PATH}')

100%|██████████| 30/30 [22:59<00:00, 45.99s/it]

Collected 30 predictions


In [4]:
results_df = pd.read_csv(OUT_PATH)
print(f'Total predictions: {len(results_df)}')
print(f'Unique lesions:    {results_df["lesion_id"].nunique()}')
print(f'\nImage mode distribution:\n{results_df["image_mode"].value_counts()}')
print(f'\ny3 distribution:\n{results_df["ground_truth"].value_counts()}')
print(f'\ny16 distribution:\n{results_df["y16"].value_counts()}')
results_df.head(10)

Saved 30 rows to /scratch/jq2uw/derm_vlms/results/gpt53_predictions_paired.csv


,id,ground_truth,image_mode,describe,classify,describe_then_classify,original_image_name,lesion_id
0,1_photo,benign,photo,On the distal lower leg and ankle (gaiter regi...,Top 3 differential diagnoses based on the imag...,**Morphologic description:** \nDistal lower l...,s-prd-667118134.jpg,534_left lower leg_no
1,1_dscope,benign,dscope,"Dermoscopic appearance:\n\n- Solitary, round t...",Most likely dermoscopic pattern: erythematous ...,**Morphologic description:** \nSolitary eryth...,s-prd-667118139.jpg,534_left lower leg_no
2,1_combined,benign,combined,Location: lower leg (pretibial area).\n\nMorph...,Most consistent morphology: **erythematous sca...,"Morphology (based on the images):\n\nSolitary,...",s-prd-667118134.jpg; s-prd-667118139.jpg,534_left lower leg_no
3,2_photo,benign,photo,On the anterior upper chest (presternal area) ...,Top 3 diagnoses:\n\n1. **Inflamed folliculitis...,"Morphologic description: \nSolitary ~4–5 mm, ...",s-prd-505211076.jpg,215_central chest _no
4,2_dscope,benign,dscope,"Solitary, round erythematous papule/nodule wit...",Top 3 differential diagnoses based on the visi...,"Description (morphology):\n\nSolitary, round e...",s-prd-505211307.jpg,215_central chest _no
5,2_combined,benign,combined,Solitary lesion on the anterior neck.\n\nClini...,Top 3 differential diagnoses based on the clin...,"Morphologic description: \nSolitary, well-cir...",s-prd-505211076.jpg; s-prd-505211307.jpg,215_central chest _no
6,3_photo,benign,photo,"Solitary, well-demarcated, round to oval eryth...","Based on the visible morphology: a solitary, w...",Morphology (based on the image):\n\n• Solitary...,s-prd-529717709.jpg,262_left buttock_no
7,3_dscope,benign,dscope,"Solitary, well-demarcated oval erythematous pl...",Based on the dermoscopic appearance: a well‑de...,"Description (morphology):\n\nSolitary, well-de...",s-prd-529718156.jpg,262_left buttock_no
8,3_combined,benign,combined,"Solitary, well-demarcated, round to oval eryth...",1. **Porokeratosis (likely disseminated superf...,Morphology (based on the images):\n\n- Solitar...,s-prd-529717709.jpg; s-prd-529718156.jpg,262_left buttock_no
9,4_photo,benign,photo,On the scalp there is a **solitary ~8–12 mm er...,"Based on the image, this appears to be a **cru...",Morphology (based on the image):\n\nSolitary ~...,s-prd-649658541.jpg,476_mid vertex_no
